# Modificación de los archivos de grafo para que los lea d-mercator

d-mercator necesita recibir la información del grafo con la forma de un archivo con dos columnas. La primera columna se corresponde con el nodo de origen de cada una de las aristas de un grafo y la segunda columan con el nodo de destino de dichas aristas.

Además, el grafo debe ser conexo (en caso contrario, d-mercator creará un nuevo archivo ".edge", con la componente gigante del grafo).

In [29]:
import networkx as nx
from pathlib import Path

In [30]:
# Horas críticas identificadas en el artículo de Beiró and Gandica et al.
HORA_CRITICA_NAT = "429624"
HORA_CRITICA_9N = "437037"
HORA_CRITICA_CH = "394721"
HORA_CRITICA_WD = "422369"

## Creación de los archivos de grafo

In [31]:
def create_graphs(MANIFESTACION,HORA_CRITICA):
    G = nx.read_gexf("../graphs/nodes_hashtag/" + MANIFESTACION + "/" + HORA_CRITICA +".gexf")

    # Componente gigante del grafo
    Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
    G = G.subgraph(Gcc[0])

    with open("graphs/" + MANIFESTACION + "/"+ HORA_CRITICA + "/" + HORA_CRITICA + ".edge", "w") as f:
        for edge in G.edges():
            f.write(edge[0] + ' ' + edge[1] + '\n')

In [32]:
# No al tarifazo
#create_graphs("nat", HORA_CRITICA_NAT)

# 9n
#create_graphs("9n", HORA_CRITICA_9N)

# Ch
create_graphs("ch", HORA_CRITICA_CH)


A continuación, hay que ejecutar el archivo dmercator_script.py desde terminal introduciendo como parámetro el archivo que debe usar: ```python3 dmercator.py```

## Creación de los archivos de grafo filtrando aristas

In [33]:
def create_filtered_graph(G, thresh_filt):
    """
    Crea un nuevo grafo a partir de un grafo original, filtrando las aristas según un umbral de peso.

    Dado un grafo `G`, esta función genera un nuevo grafo en el que solo se mantienen las aristas cuyo peso
    es mayor o igual al umbral especificado. Las aristas con peso inferior al umbral son eliminadas.

    Parámetros:
    -----------
    G : networkx.Graph
        El grafo original del cual se va a filtrar.

    thresh_filt : float
        El umbral de peso. Solo se conservarán las aristas cuyo peso sea mayor o igual a este umbral.

    Retorna:
    --------
    networkx.Graph
        Un nuevo grafo que contiene solo las aristas con peso mayor o igual al umbral especificado.
    """
    H = nx.Graph()
    # Añade nodos del grafo original al nuevo grafo
    H.add_nodes_from(G.nodes())

    # Añade aristas que cumplen con el umbral de peso
    for u, v, data in G.edges(data=True):
        if data['weight'] >= thresh_filt:
            H.add_edge(u, v, **data)
    return H

In [34]:
def get_reduction_graph(MANIFESTACION, HORA_CRITICA, umbral):
    G = nx.read_gexf("../graphs/nodes_hashtag/" + MANIFESTACION + "/" + HORA_CRITICA + ".gexf")
    nodos_original = G.number_of_nodes()
    aristas_original = G.number_of_edges()
    G = create_filtered_graph(G, umbral)

    # Componente gigante del grafo
    Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
    G = G.subgraph(Gcc[0])

    directory = "graphs/" + MANIFESTACION + "/" + HORA_CRITICA + "/" + str(umbral) + "/"

    Path(directory).mkdir(parents=True, exist_ok=True)

    with open(directory + HORA_CRITICA + ".edge", "w") as f:
        for edge in G.edges():
            f.write(edge[0] + ' ' + edge[1] + '\n')
    print("Grafo (umbral", str(umbral), "reducido a", round(G.number_of_nodes()/nodos_original*100,2), "% en nodos y ", round(G.number_of_edges()/aristas_original*100, 2), "% en aristas.")


In [36]:
umbral_nat = 5
umbral_9n = 5
umbral_ch = 1
umbral_wd = 2

"""# No al tarifazo
get_reduction_graph("nat", HORA_CRITICA_NAT, umbral_nat)

# 9n
get_reduction_graph("9n", HORA_CRITICA_9N, umbral_9n)"""

"""# ch
get_reduction_graph("ch", HORA_CRITICA_CH, umbral_ch)"""

# wd
get_reduction_graph("wd", HORA_CRITICA_WD, umbral_wd)

Grafo (umbral 2 reducido a 10.42 % en nodos y  4.69 % en aristas.
